In [1]:
from pathlib import Path
from typing import List, Tuple, Dict
import json, random, itertools, statistics as st
import seqeval

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import KFold, train_test_split
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import BallTree
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity, pairwise_distances
from sklearn.cluster import KMeans

import torch
from transformers import (
    AutoTokenizer,
        AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

from seqeval.metrics import f1_score, classification_report
from evaluate import load as load_metric
from tqdm.auto import tqdm

c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Configuração e Verificação Inicial

In [2]:
SEED_GLOBAL = 42
random.seed(SEED_GLOBAL)
np.random.seed(SEED_GLOBAL)
torch.manual_seed(SEED_GLOBAL)

CSV_PATH = Path("data/cachacaNER.csv")  # ajuste se necessário
MODEL_NAME = "pierreguillou/bert-base-cased-pt-ner"

df = pd.read_csv(CSV_PATH)

In [3]:
for cand in ["sentence_id", "sentence", "sent_id"]:
    if cand in df.columns:
        SENT_COL = cand
        break
else:
    raise KeyError(f"Coluna de sentença não encontrada em {list(df.columns)}")


def sent_to_record(sent_id, g):
    return {
        "sentence_id": int(sent_id),
        "tokens": g["token"].tolist(),
        "ner_tags": g["tag"].tolist(),
    }


In [4]:
records = [sent_to_record(i, g) for i, g in df.groupby(SENT_COL, sort=False)]
cachaca_full = Dataset.from_list(records)

In [5]:
cachaca_full

Dataset({
    features: ['sentence_id', 'tokens', 'ner_tags'],
    num_rows: 13628
})

In [6]:
# lista de rótulos (ordem alfabética garante consistência entre runs)
label_list = sorted({lab for sent in cachaca_full["ner_tags"] for lab in sent})
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
NUM_LABELS = len(label_list)

In [7]:
id2label

{0: 'B-CARACTERISTICA_SENSORIAL_AROMA',
 1: 'B-CARACTERISTICA_SENSORIAL_CONSISTÊNCIA',
 2: 'B-CARACTERISTICA_SENSORIAL_COR',
 3: 'B-CARACTERISTICA_SENSORIAL_SABOR',
 4: 'B-CLASSIFICACAO_BEBIDA',
 5: 'B-EQUIPAMENTO_DESTILACAO',
 6: 'B-GRADUACAO_ALCOOLICA',
 7: 'B-NOME_BEBIDA',
 8: 'B-NOME_LOCAL',
 9: 'B-NOME_ORGANIZACAO',
 10: 'B-NOME_PESSOA',
 11: 'B-PRECO',
 12: 'B-RECIPIENTE_ARMAZENAMENTO',
 13: 'B-TEMPO',
 14: 'B-TEMPO_ARMAZENAMENTO',
 15: 'B-TIPO_MADEIRA',
 16: 'B-VOLUME',
 17: 'I-CARACTERISTICA_SENSORIAL_AROMA',
 18: 'I-CARACTERISTICA_SENSORIAL_CONSISTÊNCIA',
 19: 'I-CARACTERISTICA_SENSORIAL_COR',
 20: 'I-CARACTERISTICA_SENSORIAL_SABOR',
 21: 'I-CLASSIFICACAO_BEBIDA',
 22: 'I-EQUIPAMENTO_DESTILACAO',
 23: 'I-GRADUACAO_ALCOOLICA',
 24: 'I-NOME_BEBIDA',
 25: 'I-NOME_LOCAL',
 26: 'I-NOME_ORGANIZACAO',
 27: 'I-NOME_PESSOA',
 28: 'I-PRECO',
 29: 'I-RECIPIENTE_ARMAZENAMENTO',
 30: 'I-TEMPO',
 31: 'I-TEMPO_ARMAZENAMENTO',
 32: 'I-TIPO_MADEIRA',
 33: 'I-VOLUME',
 34: 'O'}

In [8]:
NUM_LABELS

35

# Splits

In [9]:
def split_standard(ds: Dataset) -> DatasetDict:
    """Usa coluna trainingTest do CSV (80/20 original)."""
    if "trainingTest" not in df.columns:
        raise ValueError("CSV não contém a coluna 'trainingTest'")
    train_ids = df.loc[df["trainingTest"] == "training", SENT_COL].unique()
    test_ids = df.loc[df["trainingTest"] == "test", SENT_COL].unique()
    return DatasetDict(
        train=ds.filter(lambda ex: ex["sentence_id"] in train_ids),
        dev=ds.filter(lambda ex: ex["sentence_id"] in test_ids),
    )

In [10]:
def random_splits(
    ds: Dataset, test_size=0.2, seeds: List[int] = range(30)
) -> List[DatasetDict]:
    triples = []
    for s in seeds:
        train, dev = ds.train_test_split(test_size=test_size, seed=s).values()
        triples.append(DatasetDict(train=train, dev=dev))
    return triples

In [11]:
# def split_heur_length(ds: Dataset, top_pct: float = 0.20) -> DatasetDict:
#     lengths = np.array([len(t) for t in ds["tokens"]])
#     thr = np.percentile(lengths, 100 * (1 - top_pct))
#     mask = lengths >= thr
#     return DatasetDict(train=ds.filter(~mask), dev=ds.filter(mask))


def split_heur_length(ds: Dataset, top_pct: float = 0.20) -> DatasetDict:
    """20 % das sentenças mais longas viram conjunto de validação (dev)."""
    lengths = np.array([len(t) for t in ds["tokens"]])
    thr = np.percentile(lengths, 100 * (1 - top_pct))  
    mask = lengths >= thr  

    dev_idx = np.where(mask)[0].tolist()  # índices → list[int]
    train_idx = np.where(~mask)[0].tolist()

    return DatasetDict(
        train=ds.select(train_idx),
        dev=ds.select(dev_idx),
    )

    # Tamanho da sentenças


# def split_heur_rare(ds: Dataset, freq_thr: int = 5) -> DatasetDict:
#     freq = Counter(w.lower() for sent in ds["tokens"] for w in sent)
#     rare = {w for w, c in freq.items() if c <= freq_thr}

#     def has_rare(example):
#         return any(w.lower() in rare for w in example["tokens"])

#     return DatasetDict(
#         train=ds.filter(lambda ex: not has_rare(ex)), dev=ds.filter(has_rare)
#     )


def split_heur_rare(ds: Dataset, freq_thr: int = 5) -> DatasetDict:
    freq = Counter(w.lower() for sent in ds["tokens"] for w in sent)
    rare = {w for w, c in freq.items() if c <= freq_thr}

    keep_dev = []
    for sent in ds["tokens"]:
        print(sent)
        keep_dev.append(any(w.lower() in rare for w in sent))

    dev_idx = [i for i, x in enumerate(keep_dev) if x]
    train_idx = [i for i, x in enumerate(keep_dev) if not x]

    return DatasetDict(
        train=ds.select(train_idx),
        dev=ds.select(dev_idx),
    )

    # Raridade dos tokens

In [12]:
def split_adversarial(ds: Dataset, pct_test: float = 0.20) -> DatasetDict:
    k = int(len(ds) * pct_test)
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    embeds = model.encode([" ".join(t) for t in ds["tokens"]], show_progress_bar=False)
    tree = BallTree(embeds, leaf_size=40)

    idx_train, idx_test = set(range(len(ds))), []
    # semente = ponto mais central
    seed_idx = np.argmax(np.linalg.norm(embeds - embeds.mean(0), axis=1))
    idx_train.remove(seed_idx)
    idx_test.append(seed_idx)
    print(k)
    while len(idx_test) < k:
        print(len(idx_test))
        dists, _ = tree.query(embeds[list(idx_train)], k=1, return_distance=True)
        nxt = list(idx_train)[int(np.argmax(dists))]
        idx_train.remove(nxt)
        idx_test.append(nxt)

    return DatasetDict(
        train=ds.select(sorted(idx_train)),
        dev=ds.select(sorted(idx_test)),
    )

    # Maximizando Wassertein Distance


def split_adversarial_fast(ds: Dataset, pct_test: float = 0.20) -> DatasetDict:
    """
    Farthest-Point Sampling aproximando Wasserstein – versão vetorizada.
    Seleciona pct_test (~20 %) das sentenças como conjunto 'dev'.
    """
    k = int(len(ds) * pct_test)
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    embeds = model.encode(
        [" ".join(t) for t in ds["tokens"]],
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,  # acelera distância euclidiana ≈ cos
    )

    n = embeds.shape[0]
    idx_all = np.arange(n)

    # 1) ponto mais "central" (norma mais distante da média)
    seed_idx = np.argmax(np.linalg.norm(embeds - embeds.mean(0), axis=1))
    selected = [seed_idx]

    # 2) vetor de distâncias mínimas a qualquer ponto já escolhido
    min_dists = np.linalg.norm(embeds - embeds[seed_idx], axis=1)

    while len(selected) < k:
        next_idx = np.argmax(min_dists)
        selected.append(next_idx)

        # atualiza min_dists com a distância ao novo ponto — tudo de uma vez
        d_new = np.linalg.norm(embeds - embeds[next_idx], axis=1)
        min_dists = np.minimum(min_dists, d_new)

    train_idx = np.setdiff1d(idx_all, selected, assume_unique=True)

    return DatasetDict(
        train=ds.select(train_idx.tolist()),
        dev=ds.select(selected),
    )

In [13]:
# def loc_split(
#     dataset: Dataset, pct_test: float = 0.20, ngram: int = 4, seed: int = 42
# ) -> DatasetDict:
#     """
#     Split baseado em baixa sobreposição léxica (4-gram Jaccard).
#     Teste = pct_test das sentenças com menor overlap em relação ao pool.
#     """
#     # 1. Texto plano por sentença
#     docs = [" ".join(toks) for toks in dataset["tokens"]]

#     # 2. Vetorizar 4-grams (binário)
#     vect = CountVectorizer(
#         analyzer="word", ngram_range=(ngram, ngram), binary=True
#     ).fit(docs)
#     X = vect.transform(docs)

#     # 3. Similaridade Jaccard aproximada com matriz binária
#     # Jaccard(A,B) = |A∩B|/|A∪B| = 1 - |AΔB|/|A∪B|
#     # Usamos: overlap = (A·Bᵀ) / (|A|+|B|-A·Bᵀ)
#     bin_counts = X.sum(axis=1).A1

#     # Para cada doc i, escolhemos vizinho + próximo (fast):
#     from sklearn.metrics.pairwise import cosine_similarity

#     # (cosine no binário ∝ |A∩B|)
#     sim = cosine_similarity(X, dense_output=False)
#     # Soma dos top-k overlaps (k=5) como score
#     k = 5
#     topk = np.zeros(len(dataset))
#     for i in range(sim.shape[0]):
#         row = sim.getrow(i).toarray()[0]
#         idx = np.argpartition(-row, range(1, k + 1))[1 : k + 1]
#         # overlap ≈ |∩|
#         inter = row[idx] * bin_counts[i]
#         uni = bin_counts[i] + bin_counts[idx] - inter
#         topk[i] = (inter / uni).mean()

#     # 4. Ordenar por overlap crescente ⇒ mais “novos” vão p/ teste
#     order = np.argsort(topk)
#     n_test = int(len(dataset) * pct_test)
#     test_idx = order[:n_test]
#     train_idx = order[n_test:]

#     return DatasetDict(
#         {"train": dataset.select(train_idx), "test": dataset.select(test_idx)}
#     )

In [14]:
def loc_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    ngram: int = 4,
    seed: int = 42,
) -> DatasetDict:
    """
    Divide por sobreposição léxica (n-gram Jaccard).
    Frações independentes para teste e validação.
    """
    docs = [" ".join(toks) for toks in dataset["tokens"]]

    vect = CountVectorizer(
        analyzer="word", ngram_range=(ngram, ngram), binary=True
    ).fit(docs)
    X = vect.transform(docs)
    bin_counts = X.sum(axis=1).A1

    sim = cosine_similarity(X, dense_output=False)
    k = 5
    topk = np.zeros(len(dataset))
    for i in range(sim.shape[0]):
        row = sim.getrow(i).toarray()[0]
        idx = np.argpartition(-row, range(1, k + 1))[1 : k + 1]
        inter = row[idx] * bin_counts[i]
        uni = bin_counts[i] + bin_counts[idx] - inter
        topk[i] = (inter / uni).mean()

    order = np.argsort(topk)  # baixo → alto overlap
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[:n_test]
    val_idx = order[n_test : n_test + n_val]
    train_idx = order[n_test + n_val :]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [15]:
def semantic_cluster_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    k: int | None = None,
    seed: int = 42,
) -> DatasetDict:
    """
    Clusters SBERT → reserva clusters distantes para test/val.
    """
    if k is None:
        k = int(np.sqrt(len(dataset)))

    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    embeddings = sbert.encode(
        [" ".join(t) for t in tqdm(dataset["tokens"])],
        batch_size=64,
        show_progress_bar=False,
    )

    km = KMeans(n_clusters=k, random_state=seed, n_init=10).fit(embeddings)
    labels = km.labels_
    centroids = km.cluster_centers_

    global_center = embeddings.mean(0, keepdims=True)
    dists = pairwise_distances(centroids, global_center).flatten()

    clusters_sorted = np.argsort(-dists)  # mais distantes primeiro
    test_clusters, val_clusters = set(), set()
    total_test = total_val = 0
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    for c in clusters_sorted:
        size = np.sum(labels == c)
        if total_test < n_test:  # preenche primeiro o teste
            test_clusters.add(c)
            total_test += size
        elif total_val < n_val:  # depois a validação
            val_clusters.add(c)
            total_val += size
        if total_test >= n_test and total_val >= n_val:
            break

    test_idx = np.where([lbl in test_clusters for lbl in labels])[0]
    val_idx = np.where([lbl in val_clusters for lbl in labels])[0]
    train_idx = np.where(
        [lbl not in test_clusters and lbl not in val_clusters for lbl in labels]
    )[0]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [16]:
def difficulty_scores(dataset: Dataset) -> np.ndarray:
    length = np.array([len(tok) for tok in dataset["tokens"]], dtype=float)
    length = (length - length.mean()) / length.std()

    dens = []
    for labels in dataset["ner_tags"]:
        non_o = sum(1 for l in labels if l != "O")
        dens.append(non_o / len(labels))
    dens = np.array(dens)
    dens = (dens - dens.mean()) / dens.std()

    ent_types, freq = [], Counter()
    for labels in dataset["ner_tags"]:
        types = [l[2:] for l in labels if l != "O"]
        ent_types.append(types[0] if types else "NONE")
    freq.update(ent_types)
    rarity = np.array([1 / freq[t] for t in ent_types])
    rarity = (rarity - rarity.mean()) / rarity.std()

    return length + dens + rarity


def reverse_curriculum_split(
    dataset: Dataset, pct_test: float = 0.20, pct_val: float = 0.10, seed: int = 42
) -> DatasetDict:
    scores = difficulty_scores(dataset)
    order = np.argsort(scores)  # easy→hard
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[-n_test:]  # hardest
    val_idx = order[-(n_test + n_val) : -n_test]
    train_idx = order[: -(n_test + n_val)]

    rng = np.random.RandomState(seed)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [17]:
def heur_len_split(dataset: Dataset,
                   pct_test: float = 0.20,
                   pct_val : float = 0.10,
                   seed: int = 42) -> DatasetDict:
    """Testa só sentenças ≥ máx(len_train)."""
    sent_lens = np.array([len(t) for t in dataset["tokens"]])
    # separação inicial train/dev (randômica estratificando por tamanho grosso)
    idx_all   = np.arange(len(dataset))
    train_idx, temp_idx = train_test_split(idx_all,
                                           test_size=pct_test + pct_val,
                                           stratify=(sent_lens//5),  # bin len
                                           random_state=seed)
    # define longo-threshold como tamanho máx. do treino
    max_train_len = sent_lens[train_idx].max()
    # test = sentenças > threshold.  Caso falte/ sobre exemplos, ajusta.
    test_idx  = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]
    # completa ou reduz para atingir pct_test
    need = int(pct_test*len(dataset)) - len(test_idx)
    if need > 0:
        test_idx.extend(resto_idx[:need])
        val_idx = resto_idx[need:]
    else:
        val_keep = int(pct_val*len(dataset))
        val_idx  = resto_idx[:val_keep]
        test_idx = test_idx[: int(pct_test*len(dataset))]
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 2) Heuristic Rare-Words ----------------------------------
def heur_rare_split(dataset: Dataset,
                    pct_test: float = 0.20,
                    pct_val : float = 0.10,
                    seed: int = 42) -> DatasetDict:
    """Testa frases que contenham palavras do quintil + raro."""
    # contagem de frequência de token
    freqs = Counter(w for toks in dataset["tokens"] for w in toks)
    # define rareza: 20 % mais raras
    thresh = np.quantile(list(freqs.values()), 0.20)
    rare_set = {w for w,c in freqs.items() if c <= thresh}
    is_rare = np.array([any(w in rare_set for w in toks)
                        for toks in dataset["tokens"]])
    rare_idx   = np.where(is_rare)[0]
    common_idx = np.where(~is_rare)[0]
    # garante proporções desejadas
    n_test = int(pct_test*len(dataset))
    n_val  = int(pct_val *len(dataset))
    rng = np.random.default_rng(seed)
    test_idx = rng.choice(rare_idx, size=min(len(rare_idx), n_test),
                          replace=False)
    resto_idx = [i for i in rare_idx if i not in test_idx] + list(common_idx)
    val_idx  = rng.choice(resto_idx, size=n_val, replace=False)
    train_idx = [i for i in resto_idx if i not in val_idx]
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 3) Standard (80-10-10) -----------------------------------
def std_split(dataset: Dataset,
              pct_test: float = 0.10,
              pct_val : float = 0.10,
              seed: int = 42) -> DatasetDict:
    """Split aleatório estratificado por comprimento (PTB-like)."""
    idx = np.arange(len(dataset))
    strat = (np.array([len(t) for t in dataset["tokens"]]) // 5)
    train_idx, temp_idx = train_test_split(idx, test_size=pct_test+pct_val,
                                          stratify=strat, random_state=seed)
    val_rel = pct_val / (pct_test+pct_val)
    val_idx, test_idx = train_test_split(temp_idx, test_size=1-val_rel,
                                         stratify=strat[temp_idx],
                                         random_state=seed)
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 4) Adversarial (approx. Wasserstein) ---------------------
def adversarial_split(dataset: Dataset,
                      pct_test: float = 0.20,
                      pct_val : float = 0.10,
                      k: int = 5,
                      seed: int = 42) -> DatasetDict:
    """
    Seleciona k frases 'mais distantes' recursivamente (BallTree + W₂)
    para compor o teste, lembrando Alg.-1 de Søgaard et al.【turn6file4】.
    """
    # SBERT embed (rápido na GPU / aceitável CPU)
    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    emb = sbert.encode([" ".join(toks) for toks in dataset["tokens"]],
                       batch_size=64, show_progress_bar=False)
    # BallTree para vizinhança eficiente
    tree = BallTree(emb, leaf_size=40)
    idx_pool = set(range(len(dataset)))
    test_idx = []
    rng = np.random.default_rng(seed)
    print(f"Selecionando {int(pct_test*len(dataset))} sentenças para teste...")
    while len(test_idx) < int(pct_test*len(dataset)):
        print(f"  {len(test_idx)} selecionadas...")
        # amostra candidata: ponto + longe do centro
        center = emb[list(idx_pool)].mean(0, keepdims=True)
        dists, _ = tree.query(center, k=len(idx_pool))
        farthest = list(idx_pool)[int(dists.argmax())]
        # pega-se farest e seus k-NN mais próximos  ⇒ aumenta diversidade
        nn = tree.query([emb[farthest]], k=k, return_distance=False)[0]
        for j in nn:
            if j in idx_pool and len(test_idx) < int(pct_test*len(dataset)):
                test_idx.append(j)
                idx_pool.remove(j)
    # retira val
    val_size = int(pct_val*len(dataset))
    val_idx  = rng.choice(list(idx_pool), size=val_size, replace=False)
    idx_pool -= set(val_idx)
    train_idx = list(idx_pool)
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [18]:
def heur_len_split(dataset: Dataset,
                   pct_test: float = 0.20,
                   pct_val : float = 0.10,
                   seed: int = 42,
                   bin_size: int = 10) -> DatasetDict:
    """
    Teste = sentenças mais longas que max(len(train)).
    Robusto a datasets pequenos: se estratificação falhar, usa split aleatório.
    """
    rng = np.random.default_rng(seed)
    idx_all   = np.arange(len(dataset))
    sent_lens = np.array([len(t) for t in dataset["tokens"]])

    # ------ 1) tenta split estratificado por baldes -------------------------
    strat = (sent_lens // bin_size)
    try:
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            stratify=strat,
            random_state=seed,
        )
    except ValueError:                       # classes com 1 amostra
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    # ------ 2) escolhe test = len > max(train) ------------------------------
    max_train_len = sent_lens[train_idx].max()
    test_idx  = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]

    # garante tamanhos exatos
    n_test_desired = int(pct_test * len(dataset))
    n_val_desired  = int(pct_val  * len(dataset))

    # completa teste se ficou pequeno
    if len(test_idx) < n_test_desired:
        extra = rng.choice(resto_idx,
                           size=n_test_desired - len(test_idx),
                           replace=False)
        test_idx.extend(extra)
        resto_idx = [i for i in resto_idx if i not in extra]

    # define validação
    val_idx  = rng.choice(resto_idx, size=n_val_desired, replace=False)
    train_idx = [i for i in idx_all if i not in test_idx and i not in val_idx]

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [19]:
def std_split(dataset,
              pct_test: float = 0.10,
              pct_val : float = 0.10,
              seed: int = 42,
              bin_size: int = 5) -> DatasetDict:
    """
    Split 80-10-10 robusto.
      • Tenta estratificar por comprimento // bin_size.
      • Se houver classes com <2 amostras, recua p/ split aleatório.
    """
    idx  = np.arange(len(dataset))
    bins = (np.array([len(t) for t in dataset["tokens"]]) // bin_size)

    try:
        train_idx, temp_idx = train_test_split(
            idx,
            test_size=pct_test + pct_val,
            stratify=bins,
            random_state=seed,
        )
    except ValueError:                       # classes muito pequenas
        train_idx, temp_idx = train_test_split(
            idx,
            test_size=pct_test + pct_val,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    # fraciona temp em val / test mantendo proporção desejada
    val_share = pct_val / (pct_test + pct_val)
    try:
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1 - val_share,
            stratify=bins[temp_idx],
            random_state=seed,
        )
    except ValueError:
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1 - val_share,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [20]:
def adversarial_split(dataset: Dataset,
                      pct_test: float = 0.20,
                      pct_val : float = 0.10,
                      k: int = 5,
                      seed: int = 42) -> DatasetDict:
    """
    Versão robusta: nunca “trava” antes de atingir n_test.
    Seleciona blocos de k sentenças mais distantes do centro iterativamente.
    """
    # -------------------------------- embeds -------------------------------
    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    emb = sbert.encode([" ".join(t) for t in dataset["tokens"]],
                       batch_size=64, show_progress_bar=False)
    tree = BallTree(emb, leaf_size=40)

    n_test = int(pct_test * len(dataset))
    n_val  = int(pct_val  * len(dataset))

    idx_pool = set(range(len(dataset)))
    test_idx = []
    rng = np.random.default_rng(seed)

    print(f"Selecionando {n_test} sentenças para teste…")
    step = 0
    while len(test_idx) < n_test and idx_pool:
        if step % 5 == 0:
            print(f"  {len(test_idx)} selecionadas…")
        step += 1

        pool_list = list(idx_pool)
        center = emb[pool_list].mean(0, keepdims=True)

        # distância euclidiana ao centro global restante
        dists, _ = tree.query(center, k=len(pool_list))
        farthest_local_idx = int(dists.argmax())
        farthest_global_idx = pool_list[farthest_local_idx]

        # número efetivo de vizinhos
        k_eff = min(k, len(idx_pool))
        nn = set(tree.query([emb[farthest_global_idx]],
                            k=k_eff,
                            return_distance=False)[0])

        # adiciona vizinhos ainda não selecionados
        for j in nn:
            if j in idx_pool and len(test_idx) < n_test:
                test_idx.append(j)
                idx_pool.remove(j)

        # Se nada foi adicionado (pode acontecer quando sobram <k únicos)
        if farthest_global_idx not in test_idx:
            test_idx.append(farthest_global_idx)
            idx_pool.remove(farthest_global_idx)

    # ------------------------ validação e treino ---------------------------
    val_idx = rng.choice(list(idx_pool), size=n_val, replace=False)
    idx_pool -= set(val_idx)
    train_idx = list(idx_pool)

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [21]:
standard_split = std_split(cachaca_full)
print('std')
# random_splt = random_splits(cachaca_full)
# print('random')
heur_len = heur_len_split(cachaca_full)
print("heur_len")
heur_rare = heur_rare_split(cachaca_full)
print("heur_rare")
advers = adversarial_split(cachaca_full)
print("advs")
loc = loc_split(cachaca_full)
print("loc")
semantic = semantic_cluster_split(cachaca_full)
print("semantic")
reverse = reverse_curriculum_split(cachaca_full)
print("reverse")

std
heur_len
heur_rare
Selecionando 2725 sentenças para teste…
  0 selecionadas…
  26 selecionadas…
  51 selecionadas…
  76 selecionadas…
  101 selecionadas…
  127 selecionadas…
  152 selecionadas…
  177 selecionadas…
  202 selecionadas…
  228 selecionadas…
  254 selecionadas…
  278 selecionadas…
  304 selecionadas…
  327 selecionadas…
  354 selecionadas…
  380 selecionadas…
  406 selecionadas…
  430 selecionadas…
  451 selecionadas…
  469 selecionadas…
  492 selecionadas…
  516 selecionadas…
  541 selecionadas…
  565 selecionadas…
  589 selecionadas…
  606 selecionadas…
  629 selecionadas…
  643 selecionadas…
  664 selecionadas…
  679 selecionadas…
  700 selecionadas…
  718 selecionadas…
  745 selecionadas…
  770 selecionadas…
  788 selecionadas…
  813 selecionadas…
  831 selecionadas…
  848 selecionadas…
  866 selecionadas…
  885 selecionadas…
  901 selecionadas…
  925 selecionadas…
  939 selecionadas…
  944 selecionadas…
  969 selecionadas…
  985 selecionadas…
  998 selecionadas…
  

C:\Users\user\AppData\Local\Temp\ipykernel_1300\4053031328.py:28: RuntimeWarning: invalid value encountered in divide
  topk[i] = (inter / uni).mean()


loc


100%|██████████| 13628/13628 [00:00<00:00, 1115012.00it/s]


semantic
reverse


# Experimentos

In [22]:



from sklearn.metrics import f1_score as skl_f1

In [23]:
def train_ner_with_split(
    dataset: Dataset,
    split: str,  # "loc" | "semantic" | "reverse" | func
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    model_ckpt: str = "Davlan/distilbert-base-multilingual-cased-ner-hrl",
    training_args_kwargs: dict | None = None,
    split_kwargs: dict | None = None,
    seed: int = 42,
):
    """
    Treina um modelo de NER usando a estratégia de split desejada.
    Retorna (trainer, métricas_test).
    """
    # 1. Escolhe função de split ------------------------------------------------
    if callable(split):
        split_fn = split
    else:
        _map = {
            "loc"      : loc_split,
            "reverse"  : reverse_curriculum_split,
            "semantic" : semantic_cluster_split,
            "heur_len" : heur_len_split,
            "heur_rare": heur_rare_split,
            "std"      : std_split,
            "advs"     : adversarial_split,
        }
        if split not in _map:
            raise ValueError(f"split='{split}' não reconhecido.")
        split_fn = _map[split]

    split_kwargs = split_kwargs or {}
    ds = split_fn(
        dataset, pct_test=pct_test, pct_val=pct_val, seed=seed, **split_kwargs
    )  # train/val/test

    # 2. Tokenizer e modelo -----------------------------------------------------
    label_list = sorted({l for labels in dataset["ner_tags"] for l in labels})
    label2id = {l: i for i, l in enumerate(label_list)}
    id2label = {i: l for l, i in label2id.items()}

    num_labels = len(label_list)
    tok = AutoTokenizer.from_pretrained(model_ckpt)
    model = AutoModelForTokenClassification.from_pretrained(
        model_ckpt,
        num_labels=num_labels,  # ← adapta o tamanho
        ignore_mismatched_sizes=True,  # ← descarta pesos velhos da head
    )
    model.config.id2label = id2label
    model.config.label2id = label2id

    # 3. Mapeamento label↔id ----------------------------------------------------
    # label_list = sorted(
    #     {l for labels in dataset["ner_tags"] for l in labels if l != "O"}
    # )

    # 4. Tokenização + alinhamento ---------------------------------------------
    # def tok_function(ex):
    #     return tok(
    #         ex["tokens"], is_split_into_words=True, truncation=True, padding=False
    #     )

    # def align_labels(ex):
    #     word_ids = ex.word_ids()
    #     labels = []
    #     for w in word_ids:
    #         if w is None:
    #             labels.append(-100)
    #         else:
    #             labels.append(label2id.get(ex["ner_tags"][w], 0))
    #     ex["labels"] = labels
    #     return ex

    # ds_tok = ds.map(tok_function, batched=True)
    # ds_tok = ds_tok.map(align_labels)

    def tokenize_and_align_labels(examples, label_all_tokens=False):
        tokenized = tok(examples["tokens"], is_split_into_words=True, truncation=True)

        labels_batch = []
        for i, word_labels in enumerate(examples["ner_tags"]):
            word_ids = tokenized.word_ids(batch_index=i)  # <- aqui sim
            label_ids = []
            previous_word_idx = None
            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)  # máscara
                elif word_idx != previous_word_idx:
                    label_ids.append(label2id[word_labels[word_idx]])
                else:
                    # marca sub-tokens; mude para `label2id[...]`
                    # se quiser repetir label em todos os sub-tokens
                    label_ids.append(
                        label2id[word_labels[word_idx]] if label_all_tokens else -100
                    )
                previous_word_idx = word_idx
            labels_batch.append(label_ids)

        tokenized["labels"] = labels_batch
        return tokenized

    ds_tok = ds.map(
        tokenize_and_align_labels, batched=True, remove_columns=ds["train"].column_names
    )
    # 5. Métrica (seqeval) ------------------------------------------------------
    seqeval = load_metric("seqeval")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        sent_preds, sent_labels = [], []   # p/ seqeval
        flat_preds, flat_labels = [], []   # p/ sklearn

        for p_row, l_row in zip(preds, labels):
            p_sent, l_sent = [], []
            for pi, li in zip(p_row, l_row):
                if li != -100:
                    lbl_true = id2label[li]
                    lbl_pred = id2label[pi]
                    p_sent.append(lbl_pred)
                    l_sent.append(lbl_true)
                    flat_preds.append(lbl_pred)
                    flat_labels.append(lbl_true)
            sent_preds.append(p_sent)
            sent_labels.append(l_sent)

        # métricas seqeval (micro F1 = overall_f1)
        seqeval_metrics = seqeval.compute(
            predictions=sent_preds,
            references=sent_labels,
        )

        # métricas sklearn
        f1_micro    = skl_f1(flat_labels, flat_preds, average="micro",    zero_division=0)
        f1_macro    = skl_f1(flat_labels, flat_preds, average="macro",    zero_division=0)
        f1_weighted = skl_f1(flat_labels, flat_preds, average="weighted", zero_division=0)

        return {
            **seqeval_metrics,            # overall_precision / recall / f1
            "f1_micro":    f1_micro,
            "f1_macro":    f1_macro,
            "f1_weighted": f1_weighted,
        }



    # 6. Args de treinamento ----------------------------------------------------
    args_defaults = dict(
        output_dir=f"ner-{split}",
        # estratégia de avaliação + salvamento
        eval_strategy="epoch",  # novo nome (4.52+)
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="overall_f1",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=5,
        seed=seed,
        report_to="none"
    )

    if training_args_kwargs:
        args_defaults.update(training_args_kwargs)
    args = TrainingArguments(**args_defaults)

    data_collator = DataCollatorForTokenClassification(tokenizer=tok, padding=True)

    # 7. Trainer ---------------------------------------------------------------
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds_tok["train"],
        eval_dataset=ds_tok["val"],
        tokenizer=tok,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # 8. Avaliação final --------------------------------------------------------
    test_metrics = trainer.evaluate(eval_dataset=ds_tok["test"])
    return trainer, test_metrics

In [ ]:
splits = ["loc", "reverse", "semantic", "heur_len", "heur_rare", "std", "advs"]

In [ ]:
results = {}
trainer_all = {}
for s in splits:
    print(f"Treinando com split: {s}")
    trainer, metrics = train_ner_with_split(cachaca_full, split=s)
    results[s] = metrics
    trainer_all[s] = trainer
    print("F1 Macro:", metrics["eval_f1_macro"])
    print("F1 Micro:", metrics["eval_f1_micro"])
    print("F1 Weighted:", metrics["eval_f1_weighted"])
    print(metrics)
    print('\n')

Treinando com split: loc


C:\Users\user\AppData\Local\Temp\ipykernel_1300\4053031328.py:28: RuntimeWarning: invalid value encountered in divide
  topk[i] = (inter / uni).mean()
Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([35]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([35, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 2725/2725 [00:00<00:00, 12607.71 examples/s]
C:\Users\user\AppData\Local\Temp\ipykernel_1300\2810120630.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
c:\U

Epoch,Training Loss,Validation Loss,Caracteristica Sensorial Aroma,Caracteristica Sensorial Consistência,Caracteristica Sensorial Cor,Caracteristica Sensorial Sabor,Classificacao Bebida,Equipamento Destilacao,Graduacao Alcoolica,Nome Bebida,Nome Local,Nome Organizacao,Nome Pessoa,Recipiente Armazenamento,Tempo,Tempo Armazenamento,Tipo Madeira,Volume,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.230700,0.148722,"{'precision': 0.56, 'recall': 0.5833333333333334, 'f1': 0.5714285714285714, 'number': 120}","{'precision': 0.8055555555555556, 'recall': 0.6590909090909091, 'f1': 0.7250000000000001, 'number': 44}","{'precision': 0.6436781609195402, 'recall': 0.7, 'f1': 0.6706586826347305, 'number': 80}","{'precision': 0.5568181818181818, 'recall': 0.392, 'f1': 0.460093896713615, 'number': 125}","{'precision': 0.8841463414634146, 'recall': 0.8529411764705882, 'f1': 0.8682634730538923, 'number': 170}","{'precision': 0.7567567567567568, 'recall': 0.5833333333333334, 'f1': 0.6588235294117648, 'number': 48}","{'precision': 0.7666666666666667, 'recall': 0.8679245283018868, 'f1': 0.8141592920353983, 'number': 53}","{'precision': 0.7388535031847133, 'recall': 0.7341772151898734, 'f1': 0.7365079365079364, 'number': 474}","{'precision': 0.8421955403087479, 'recall': 0.9229323308270677, 'f1': 0.8807174887892377, 'number': 532}","{'precision': 0.5660377358490566, 'recall': 0.625, 'f1': 0.594059405940594, 'number': 144}","{'precision': 0.868020304568528, 'recall': 0.8860103626943006, 'f1': 0.876923076923077, 'number': 193}","{'precision': 0.9539473684210527, 'recall': 0.8682634730538922, 'f1': 0.9090909090909091, 'number': 167}","{'precision': 0.8636363636363636, 'recall': 0.9047619047619048, 'f1': 0.8837209302325582, 'number': 189}","{'precision': 0.9071428571428571, 'recall': 0.9548872180451128, 'f1': 0.9304029304029303, 'number': 133}","{'precision': 0.9252669039145908, 'recall': 0.8873720136518771, 'f1': 0.9059233449477352, 'number': 293}","{'precision': 0.9723756906077348, 'recall': 0.9887640449438202, 'f1': 0.9805013927576601, 'number': 178}",0.811761,0.816174,0.813961,0.963127,0.963127,0.788513,0.962038
2,0.054500,0.149577,"{'precision': 0.48484848484848486, 'recall': 0.6666666666666666, 'f1': 0.5614035087719298, 'number': 120}","{'precision': 0.8823529411764706, 'recall': 0.6818181818181818, 'f1': 0.7692307692307693, 'number': 44}","{'precision': 0.7073170731707317, 'recall': 0.725, 'f1': 0.7160493827160495, 'number': 80}","{'precision': 0.5168539325842697, 'recall': 0.368, 'f1': 0.4299065420560748, 'number': 125}","{'precision': 0.842391304347826, 'recall': 0.9117647058823529, 'f1': 0.8757062146892656, 'number': 170}","{'precision': 0.7719298245614035, 'recall': 0.9166666666666666, 'f1': 0.838095238095238, 'number': 48}","{'precision': 0.8103448275862069, 'recall': 0.8867924528301887, 'f1': 0.8468468468468469, 'number': 53}","{'precision': 0.7634854771784232, 'recall': 0.7763713080168776, 'f1': 0.7698744769874477, 'number': 474}","{'precision': 0.8470790378006873, 'recall': 0.9266917293233082, 'f1': 0.8850987432675045, 'number': 532}","{'precision': 0.5153061224489796, 'recall': 0.7013888888888888, 'f1': 0.5941176470588234, 'number': 144}","{'precision': 0.8702702702702703, 'recall': 0.8341968911917098, 'f1': 0.851851851851852, 'number': 193}","{'precision': 0.927710843373494, 'recall': 0.9221556886227545, 'f1': 0.924924924924925, 'number': 167}","{'precision': 0.8704663212435233, 'recall': 0.8888888888888888, 'f1': 0.8795811518324607, 'number': 189}","{'precision': 0.9703703703703703, 'recall': 0.9849624060150376, 'f1': 0.9776119402985074, 'number': 133}","{'precision': 0.8532110091743119, 'recall': 0.9522184300341296, 'f1': 0.9, 'number': 293}","{'precision': 0.9672131147540983, 'recall': 0.9943820224719101, 'f1': 0.9806094182825486, 'number': 178}",0.799230,0.846755,0.822307,0.964653,0.964653,0.802661,0.964358
3,0.035900,0.138320,"{'precision': 0.525, 'recall': 0.7, 'f1': 0.6, 'number': 1

c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warn

c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


...

F1 Macro: 0.9303260747729224
F1 Micro: 0.9855432755965088
F1 Weighted: 0.9856299073740179
{'eval_loss': 0.0611397922039032, 'eval_CARACTERISTICA_SENSORIAL_AROMA': {'precision': 0.7903225806451613, 'recall': 0.8546511627906976, 'f1': 0.8212290502793295, 'number': 172}, 'eval_CARACTERISTICA_SENSORIAL_CONSISTÊNCIA': {'precision': 0.926829268292683, 'recall': 0.8444444444444444, 'f1': 0.8837209302325582, 'number': 45}, 'eval_CARACTERISTICA_SENSORIAL_COR': {'precision': 0.8416666666666667, 'recall': 0.8782608695652174, 'f1': 0.8595744680851064, 'number': 115}, 'eval_CARACTERISTICA_SENSORIAL_SABOR': {'precision': 0.7672955974842768, 'recall': 0.7672955974842768, 'f1': 0.7672955974842768, 'number': 159}, 'eval_CLASSIFICACAO_BEBIDA': {'precision': 0.9383561643835616, 'recall': 0.9785714285714285, 'f1': 0.9580419580419579, 'number': 280}, 'eval_EQUIPAMENTO_DESTILACAO': {'precision': 0.9090909090909091, 'recall': 0.9375, 'f1': 0.923076923076923, 'number': 64}, 'eval_GRADUACAO_ALCOOLICA': {'preci

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([35]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([35, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 2726/2726 [00:00<00:00, 10038.42 examples/s]
C:\Users\user\AppData\Local\Temp\ipykernel_1300\2810120630.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as t

Epoch,Training Loss,Validation Loss,Caracteristica Sensorial Aroma,Caracteristica Sensorial Consistência,Caracteristica Sensorial Cor,Caracteristica Sensorial Sabor,Classificacao Bebida,Equipamento Destilacao,Graduacao Alcoolica,Nome Bebida,Nome Local,Nome Organizacao,Nome Pessoa,Preco,Recipiente Armazenamento,Tempo,Tempo Armazenamento,Tipo Madeira,Volume,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.255700,0.078187,"{'precision': 0.5529411764705883, 'recall': 0.746031746031746, 'f1': 0.6351351351351352, 'number': 63}","{'precision': 0.9375, 'recall': 0.6818181818181818, 'f1': 0.7894736842105263, 'number': 22}","{'precision': 0.7441860465116279, 'recall': 0.8205128205128205, 'f1': 0.7804878048780488, 'number': 39}","{'precision': 0.7307692307692307, 'recall': 0.4578313253012048, 'f1': 0.5629629629629629, 'number': 83}","{'precision': 0.8636363636363636, 'recall': 0.95, 'f1': 0.9047619047619048, 'number': 120}","{'precision': 0.8157894736842105, 'recall': 0.9117647058823529, 'f1': 0.861111111111111, 'number': 34}","{'precision': 0.9911504424778761, 'recall': 1.0, 'f1': 0.9955555555555555, 'number': 112}","{'precision': 0.7993630573248408, 'recall': 0.856655290102389, 'f1': 0.827018121911038, 'number': 293}","{'precision': 0.9096774193548387, 'recall': 0.9701834862385321, 'f1': 0.9389567147613763, 'number': 436}","{'precision': 0.8651685393258427, 'recall': 0.8461538461538461, 'f1': 0.8555555555555556, 'number': 91}","{'precision': 0.9418604651162791, 'recall': 0.9204545454545454, 'f1': 0.9310344827586208, 'number': 88}","{'precision': 0.9306930693069307, 'recall': 1.0, 'f1': 0.9641025641025642, 'number': 94}","{'precision': 0.9279279279279279, 'recall': 0.9626168224299065, 'f1': 0.944954128440367, 'number': 107}","{'precision': 0.9393939393939394, 'recall': 0.9465648854961832, 'f1': 0.9429657794676807, 'number': 131}","{'precision': 0.96, 'recall': 0.9836065573770492, 'f1': 0.97165991902834, 'number': 122}","{'precision': 0.9362549800796812, 'recall': 0.9710743801652892, 'f1': 0.9533468559837728, 'number': 242}","{'precision': 0.9922779922779923, 'recall': 0.9734848484848485, 'f1': 0.9827915869980879, 'number': 264}",0.893035,0.920120,0.906375,0.978381,0.978381,0.864998,0.977988
2,0.066300,0.072784,"{'precision': 0.5729166666666666, 'recall': 0.873015873015873, 'f1': 0.6918238993710691, 'number': 63}","{'precision': 0.9444444444444444, 'recall': 0.7727272727272727, 'f1': 0.85, 'number': 22}","{'precision': 0.8421052631578947, 'recall': 0.8205128205128205, 'f1': 0.8311688311688312, 'number': 39}","{'precision': 0.6666666666666666, 'recall': 0.7951807228915663, 'f1': 0.7252747252747251, 'number': 83}","{'precision': 0.9083969465648855, 'recall': 0.9916666666666667, 'f1': 0.9482071713147411, 'number': 120}","{'precision': 0.8333333333333334, 'recall': 0.8823529411764706, 'f1': 0.8571428571428571, 'number': 34}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 112}","{'precision': 0.8938356164383562, 'recall': 0.8907849829351536, 'f1': 0.8923076923076922, 'number': 293}","{'precision': 0.952914798206278, 'recall': 0.9747706422018348, 'f1': 0.9637188208616779, 'number': 436}","{'precision': 0.8080808080808081, 'recall': 0.8791208791208791, 'f1': 0.8421052631578948, 'number': 91}","{'precision': 0.9230769230769231, 'recall': 0.9545454545454546, 'f1': 0.9385474860335197, 'number': 88}","{'precision': 0.9306930693069307, 'recall': 1.0, 'f1': 0.9641025641025642, 'number': 94}","{'precision': 0.9454545454545454, 'recall': 0.9719626168224299, 'f1': 0.9585253456221198, 'number': 107}","{'precision': 0.946969696969697, 'recall': 0.9541984732824428, 'f1': 0.9505703422053233, 'number': 131}","{'precision': 0.9758064516129032, 'recall': 0.9918032786885246, 'f1': 0.983739837398374, 'number': 122}","{'precision': 0.9755102040816327, 'recall': 0.987603305785124, 'f1': 0.9815195071868583, 'number': 242}","{'precision': 0.9961240310077519, 'recall': 0.9734848484848485, 'f1': 0.9846743295019156

c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\user\Documents\mestrado\ner_splits\ner_splits\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warn

F1 Macro: 0.8729360209592214
F1 Micro: 0.9750559376812795
F1 Weighted: 0.9748498292036873
{'eval_loss': 0.10891634225845337, 'eval_CARACTERISTICA_SENSORIAL_AROMA': {'precision': 0.8208333333333333, 'recall': 0.8277310924369747, 'f1': 0.8242677824267782, 'number': 238}, 'eval_CARACTERISTICA_SENSORIAL_CONSISTÊNCIA': {'precision': 0.8727272727272727, 'recall': 0.8275862068965517, 'f1': 0.8495575221238938, 'number': 58}, 'eval_CARACTERISTICA_SENSORIAL_COR': {'precision': 0.8548387096774194, 'recall': 0.8346456692913385, 'f1': 0.844621513944223, 'number': 127}, 'eval_CARACTERISTICA_SENSORIAL_SABOR': {'precision': 0.7528735632183908, 'recall': 0.7401129943502824, 'f1': 0.7464387464387464, 'number': 177}, 'eval_CLASSIFICACAO_BEBIDA': {'precision': 0.8602150537634409, 'recall': 0.975609756097561, 'f1': 0.9142857142857143, 'number': 246}, 'eval_EQUIPAMENTO_DESTILACAO': {'precision': 0.8611111111111112, 'recall': 0.96875, 'f1': 0.911764705882353, 'number': 64}, 'eval_GRADUACAO_ALCOOLICA': {'prec